In [ ]:
FastAPI — это современный, быстрый (high-performance) веб-фреймворк на Python, который 
позволяет легко и быстро создавать API. Он использует асинхронные функции, чтобы обеспечить
высокую производительность, и активно использует Pydantic для работы с 
данными и валидации. Давайте разберем ключевые концепции, которые вы упомянули:

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float
    in_stock: bool = True

@app.post("/items/")
async def create_item(item: Item):
    return item


In [ ]:
/docs
/redoc

In [ ]:
1. Path Operations

In [ ]:
Path Operations (или "операции пути") — это основные точки входа в ваше приложение, которые 
соответствуют различным HTTP-методам (GET, POST, PUT, DELETE и т.д.) и путям (routes). В FastAPI 
каждая операция пути представлена функцией Python,
которая ассоциируется с определенным маршрутом и HTTP-методом.

In [ ]:
from fastapi import FastAPI
app ´FastAPI()

@app.get("/items/{item_id}")
async def read_item(item_id: int):
    return {"item_id": item_id}

In [ ]:
@app.get("/items/")
async def read_item(skip: int = 0, limit: int = 10):
    return {"skip": skip, "limit": limit}
# /items/?skip=1&limit=1

In [ ]:
@app.put("/items/{item_id}")
async def update_item(item_id: int, q: str | None = None, item: Item = None):
    return {"item_id": item_id, "q": q, "item": item}


In [ ]:
from fastapi import Path, Query

@app.get("/items/{item_id}")
async def read_item(
    item_id: int = Path(..., ge=1),
    q: str = Query(default=None, max_length=50)
):
    return {"item_id": item_id, "q": q}


In [ ]:
2. Request <-> Response, Models
FastAPI использует Pydantic для управления данными и их валидации, что позволяет легко определять 
схемы данных (модели) для запросов и ответов.
Пример модели:


In [ ]:
from pydantic import BaseModel

class Item(BaseModel):
    name: str
    description: str | None = None
    price: float
    tax: float | None = None

@app.post("/items/")
async def create_item(item: Item):
    return item

In [ ]:
@app.get("/items/{item_id}", response_model=Item)
async def read_item(item_id: int):
    return Item(name="Book", price=9.99, is_offer=True)


In [ ]:
class ItemCreate(BaseModel):
    name: str
    price: float

class ItemOut(BaseModel):
    id: int
    name: str
    price: float

@app.post("/items/", response_model=ItemOut)
async def create_item(item: ItemCreate):
    return ItemOut(id=123, name=item.name, price=item.price)


In [ ]:
from fastapi.responses import JSONResponse

@app.get("/custom")
async def custom_response():
    return JSONResponse(content={"message": "ok"}, status_code=202)


3. Pydantic
Pydantic — это библиотека для управления и валидации данных в Python. Она используется в FastAPI 
для создания моделей данных, которые затем используются для валидации и сериализации данных.
Основные возможности Pydantic:
•	Валидация входящих данных.
•	Автоматическое преобразование данных.
•	Документирование моделей.
•	Поддержка аннотаций типов Python для указания структуры данных.
Пример модели с валидацией:


In [ ]:
from pydantic import BaseModel, Field

class User(BaseModel):
    username: str = Field(..., min_length=3, max_length=50)
    email: str
    age: int|None = None

In [ ]:
#models.py
from pydantic import BaseModel

class Item(BaseModel):
    name: str
    description: str | None = None
    price: float
    in_stock: bool = True

//main.py
from models import Item
from fastapi import FastAPI
@app.post("/items/")
def create_item(item: Item):
    return {"item_created": item}
#{
#    "name": "sdf",
#    "price": 50
#}



4. Dependencies
Зависимости (Dependencies) в FastAPI позволяют модульно разделять и управлять логикой, 
которая может быть повторно использована в различных операциях пути. Это может включать в 
себя валидацию данных, проверку аутентификации, подключение к базе данных и многое другое.
Пример использования зависимости:


In [ ]:
from fastapi import Depends, FastAPI
app = FastAPI()
def get_query(q: str | None = None):
    return q

@app.get("/items/")
async def read_items(q: str = Depends(get_query)):
    return {"q": q}

In [ ]:
from fastapi import HTTPException, status

def get_current_user(token: str = Depends(oauth2_scheme)):
    if token != "sekret-token":
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED)
    return {"username": "alice"}
@app.get("/users/me")
async def read_current_user(user: dict = Depends(get_current_user)):
    return user



In [ ]:
def pagination(skip: int = 0, limit: int = 10):
    return {"skip": skip, "limit": limit}

@app.get("/books/")
async def list_books(pagination: dict = Depends(pagination)):
    return pagination


In [ ]:
5. Развертывание
FastAPI поддерживает различные методы развертывания. Один из самых распространенных способов 
развертывания — это использование Uvicorn, ASGI-сервера, который хорошо работает с FastAPI.
Простой пример развертывания через Uvicorn:


In [ ]:
unicorn main:app --reload

In [ ]:
Подведение итогов:
•	Path Operations: Это функции, связанные с маршрутами и HTTP-методами.
•	Request<->Response, Models: Использование Pydantic для валидации и управления данными через модели.
•	Pydantic: Основной инструмент для работы с данными, обеспечивает валидацию и сериализацию.
•	Dependencies: Механизм для управления повторно используемой логикой.
•	Развертывание: Обычно осуществляется с использованием Uvicorn, Gunicorn и прокси-сервера.
FastAPI — мощный и гибкий инструмент для создания API, и с его помощью можно легко создавать
масштабируемые и безопасные веб-приложения.


In [ ]:
from fastapi import FastAPI

app = FastAPI(
    title="My API",
    description="This is my awesome API that does great things.",
    version="1.0.0",
    docs_url="/docs",  # Путь к документации Swagger
    redoc_url="/redoc",  # Путь к документации ReDoc
)

app = FastAPI(docs_url=None)

